# Asistente de reglas del SRD 5.2.1

Ejecuta el código del repositorio. Requiere el secreto `ANTHROPIC_API_KEY`.

## 1. Preparación

In [7]:
import os
if os.path.exists('/content/dnd_assistant'):
    !cd /content/dnd_assistant && git pull -q
else:
    !git clone -q https://github.com/MarioGavin/dnd_assistant.git /content/dnd_assistant
%cd /content/dnd_assistant
!pip -q install pymupdf rank-bm25 snowballstemmer anthropic pytest

# La clave de los Secretos pasa al entorno para que la lean los scripts
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

/content/dnd_assistant


## 2. Tests

In [8]:
!python -m pytest -q

........................................................................ [ 56%]
........................................................                 [100%]
128 passed in 13.52s


## 3. Consultas

In [9]:
!python -m asistente.responder "Tengo 2 niveles de cansancio, ¿qué me pasa al atacar?"

Búsqueda: Tengo 2 niveles de cansancio, ¿qué me pasa al atacar? Cansancio (agotamiento); niveles de agotamiento; penalizador al d20 por agotamiento; efectos del agotamiento en tiradas de ataque; tiradas de d20; prueba de característica; tirada de salvación; estado agotado; reducción de niveles de agotamiento; descanso largo; descanso corto; reglas de estados 

Con 2 niveles de cansancio, cada prueba con d20 se reduce en 2 × 2 = 4 [1]. Como una tirada de ataque es una prueba con d20 [10][17], tus ataques tienen −4 al resultado del d20 [1][10].

Cambio respecto a 2014: en 2014, el cansancio de nivel 2 reducía la velocidad a la mitad y daba desventaja en pruebas de característica, pero no afectaba a las tiradas de ataque [2]. 

Fragmentos que recibió el modelo:
  [1] SRD 5.2.1 ES · Glosario de reglas › Definiciones de las reglas › Cansancio [estado] · p. 197
  [2] SRD 5.1 ES · Apéndice MJ-A: Estados › Cansancio · p. 389 · OBSOLETO (reglas de 2014)
  [3] SRD 5.2.1 ES · Clases › Explorador 

In [ ]:
!python -m asistente.responder "Dame la CA y los PG del contemplador"

In [ ]:
!python -m asistente.responder "Hemos pillado por sorpresa a unos bandidos, ¿pierden su primer turno?"

In [ ]:
# Validador de fichas: sin modelo, sin coste
!python -m asistente.fichas

## 4. Evaluación de la recuperación: BM25

In [ ]:
!python -m asistente.evaluar --solo-recuperacion --sin-vectores
!python -m asistente.evaluar --solo-recuperacion --sin-vectores --casos casos/consultas_contraste.json

## 5. Evaluación completa (consultas y fichas)

Tres ejecuciones por modelo: la API no admite `temperature`, así que la variabilidad se mide repitiendo.

In [ ]:
!python -m asistente.evaluar --modelo claude-haiku-5-5 --repeticiones 3

In [ ]:
!python -m asistente.evaluar --modelo claude-sonnet-5-5 --repeticiones 3

In [ ]:
# Haiku con la pregunta reformulada antes de buscar (DECISIONES.md, fila 14)
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --repeticiones 3

In [ ]:
# La primera es la configuración entregada (reformular + contexto ampliado, filas 14 y 19); las otras, opciones medidas
# contexto ampliado (60 candidatos, 20 fragmentos, 2 de 2014 y tablas del padre) y tablas del apartado padre (fallo 24)
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --contexto-amplio --repeticiones 3
!python -m asistente.evaluar --modelo claude-haiku-5-5 --tablas-padre --repeticiones 3
!python -m asistente.evaluar --modelo claude-haiku-5-5 --reformular --tablas-padre --repeticiones 3

In [ ]:
!python -m asistente.evaluar --tabla

## 6. Evaluación de la recuperación: combinada (opcional, GPU)

In [ ]:
!pip -q install sentence-transformers
!python -m asistente.evaluar --solo-recuperacion

In [ ]:
!python -m asistente.evaluar --solo-recuperacion --modelo-emb BAAI/bge-m3

## 6b. Modelo local: Qwen3 (opcional, GPU)

Antes: Entorno de ejecución → Cambiar tipo de entorno → una GPU (A100 o L4: 16 bits sin cuantizar; T4: el 8B y el 14B en 4 bits), y volver a ejecutar la sección 1. Mismo prompt y mismo contexto que `haiku-5.5` (sin reformular, contexto normal), sin razonamiento. La GPU, el tipo de datos y si se cuantizó quedan en `resumen.carga_local`.

In [ ]:
!pip -q install -U "transformers>=4.51" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Prueba de tiempo con 2 consultas. Si cada una pasa de ~60 s, no lances la evaluación completa
import gc, torch
from asistente import responder as R
for p in ["Mi pícaro se cae desde 12 metros. ¿Cuánto daño recibe?",
          "Dame la CA, los PG y los ataques del contemplador para un encuentro de nivel 5."]:
    r = R.responder(p, modelo=R.MODELO_LOCAL)
    print(r["segundos"], "s ·", r["tokens_entrada"], "+", r["tokens_salida"], "tokens ·", r["stop_reason"])
    print(r["respuesta"][:400], "\n")
R._locales.clear(); gc.collect(); torch.cuda.empty_cache()   # libera la GPU para la celda siguiente

In [ ]:
# Evaluación completa: 1 ejecución de los 20 casos
!python -m asistente.evaluar --modelo Qwen/Qwen3-4B --repeticiones 1

In [ ]:
!python -m asistente.evaluar --tabla

### Qwen3-8B (el modelo del plan)

En una T4 se carga en 4 bits (NF4); en L4 o A100, en 16 bits. La primera vez descarga unos 16 GB.

In [10]:
!pip -q install -U "transformers>=4.51" accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [11]:
# Prueba de tiempo del 8B con 2 consultas. En A100 o L4 debe salir 'cuantizado_4bits': False
import gc, torch
from asistente import responder as R
for p in ["Mi pícaro se cae desde 12 metros. ¿Cuánto daño recibe?",
          "Dame la CA, los PG y los ataques del contemplador para un encuentro de nivel 5."]:
    r = R.responder(p, modelo="Qwen/Qwen3-8B")
    print(r["segundos"], "s ·", r["tokens_entrada"], "+", r["tokens_salida"], "tokens ·", r["stop_reason"])
    print(r["respuesta"][:400], "\n")
print(R.CARGA_LOCAL)
R._locales.clear(); gc.collect(); torch.cuda.empty_cache()   # libera la GPU para la celda siguiente

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

7.01 s · 3774 + 73 tokens · end_turn
Una criatura que sufra una caída recibe 1d6 de daño contundente por cada 3 m que haya caído antes de golpearse contra el suelo, hasta un máximo de 20d6. Desde 12 metros, recibiría 4d6 de daño contundente [10]. 

2.16 s · 3056 + 14 tokens · end_turn
No está en el SRD 5.2.1. 

{'Qwen/Qwen3-8B': {'gpu': 'NVIDIA L4', 'memoria_gb': 23.7, 'dtype': 'bfloat16', 'cuantizado_4bits': True}}


In [12]:
!python -m asistente.evaluar --modelo Qwen/Qwen3-8B --repeticiones 1
!python -m asistente.evaluar --tabla


=== qwen3-8b · ejecución 1 de 1
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 399/399 [00:04<00:00, 97.44it/s] 
  ok  N01   7.62 s  'Veredicto: ilegal.\n\nEl veredicto es ilegal. La ficha utiliza el método "coste_en_puntos", '
  MAL N02   6.68 s  'Una criatura que sufra una caída recibirá 1d6 de daño contundente por cada 3 m que haya ca'
  ok  N03   3.66 s  'Un guerrero gana 6 + modificador por Constitución de puntos de golpe al subir de nivel si '
  MAL N04    8.0 s  'La CD de salvación para los conjuros de un clérigo se calcula como 8 + nivel del conjuro +'
  ok  N05    6.1 s  'La CD es de 10 o la mitad del daño sufrido (redondeando hacia abajo), lo que sea mayor, ha'
  ok  N06   7.01 s  'Veredicto: legal.\n\nLa ficha es legal.  \n- El conjunto estándar es correcto [1].  \n- Los au'
  ok  N07   7.32 s  'Veredicto: ilegal.\n\nEl veredicto es **ilegal**.  \n\n- **pg**: El valor de puntos de golpe s'
  MAL N08   2.84 s  'Sí, un mago de nivel 3 p

### Qwen3-14B (opcional, solo con A100: ~30 GB en 16 bits)

In [ ]:
!python -m asistente.evaluar --modelo Qwen/Qwen3-14B --repeticiones 1
!python -m asistente.evaluar --tabla

## 7. Resultados

In [13]:
!zip -qr resultados.zip resultados
from google.colab import files
files.download('resultados.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>